# MCS603 — Data Analysis Module
## Notebook 3: Matplotlib & Seaborn — Data Visualisation

---
### Learning Objectives
- Master the Matplotlib Figure/Axes object model
- Create distribution plots: histogram, KDE, boxplot, violin, strip
- Build correlation heatmaps with annotations
- Design before/after comparison charts
- Use Seaborn for statistical visualisations with minimal code
- Compose multi-panel publication-quality figures

### Setup
```bash
pip install matplotlib seaborn
```

### Outline
1. Matplotlib Fundamentals  
2. Distribution Plots  
3. Relationships & Scatter  
4. Correlation Heatmaps  
5. Before / After Comparisons  
6. Seaborn Statistical Plots  
7. Multi-Panel Dashboard  
8. Exercises  

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy.stats.mstats import winsorize

# Global style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight"})

REGION_COLORS = {
    "West Africa":    "#E74C3C",
    "East Africa":    "#3498DB",
    "North Africa":   "#2ECC71",
    "Central Africa": "#9B59B6",
    "Southern Africa":"#F39C12",
}

In [ ]:
try:
    df = pd.read_csv("africa_health_clean.csv")
    print("Loaded from africa_health_clean.csv")
except FileNotFoundError:
    # Minimal fallback
    np.random.seed(42)
    N = 54
    countries = [
        "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
        "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
        "Madagascar","Côte d'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
        "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
        "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
        "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
        "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
        "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
    ]
    regions = (["West Africa"]*16+["East Africa"]*14+["North Africa"]*6+["Central Africa"]*8+["Southern Africa"]*10)
    df = pd.DataFrame({
        "country"           :countries,"region":regions,
        "life_expectancy"   :np.clip(np.random.normal(63,8,N),45,80),
        "infant_mortality"  :np.clip(np.random.exponential(35,N),5,120),
        "maternal_mortality":np.clip(np.random.exponential(350,N),20,2000),
        "hiv_prevalence"    :np.clip(np.random.exponential(3.5,N),0.1,28),
        "health_expenditure":np.clip(np.random.normal(5.2,2.1,N),1,12),
        "gdp_per_capita"    :np.clip(np.exp(np.random.normal(7.2,1.2,N)),300,15000),
        "vaccination_dtp3"  :np.clip(np.random.normal(78,18,N),20,99),
        "water_access"      :np.clip(np.random.normal(68,22,N),15,99),
        "malaria_incidence" :np.clip(np.random.exponential(120,N),0,450),
    })
    df["log_gdp"] = np.log1p(df["gdp_per_capita"])

print(df.shape)
df.head(3)

---
## 1. Matplotlib Fundamentals

Matplotlib has two interfaces:
- **pyplot interface** (`plt.plot(...)`) — quick, implicit
- **Object-oriented (OO)** (`fig, ax = plt.subplots()`) — explicit, recommended

```
Figure
 └── Axes  (one or many)
      ├── Title, xlabel, ylabel
      ├── Tick labels & tick locators
      └── Artists (lines, patches, texts)
```

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

x = df["gdp_per_capita"]
y = df["life_expectancy"]

scatter = ax.scatter(x, y, c=df["region"].map(
    {r: i for i, r in enumerate(df["region"].unique())}),
    cmap="Set1", s=60, alpha=0.8, edgecolors="white", linewidths=0.5)

# Anatomy labels
ax.set_title("Life Expectancy vs. GDP per Capita\n(Africa, 2022)", fontsize=13, fontweight="bold")
ax.set_xlabel("GDP per Capita (USD)", fontsize=11)
ax.set_ylabel("Life Expectancy (years)", fontsize=11)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Trend line
m, b = np.polyfit(x, y, 1)
xline = np.linspace(x.min(), x.max(), 200)
ax.plot(xline, m*xline + b, "--", color="#2C3E50", linewidth=1.5, label="Linear trend")
ax.legend(fontsize=9)

# Annotate top countries
for _, row in df.nlargest(3, "life_expectancy").iterrows():
    ax.annotate(row["country"],
                xy=(row["gdp_per_capita"], row["life_expectancy"]),
                xytext=(10, 5), textcoords="offset points", fontsize=8,
                arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))

plt.tight_layout()
plt.savefig("scatter_gdp_le.png")
plt.show()

---
## 2. Distribution Plots

In [ ]:
# Histogram + KDE overlay
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

plot_cols = [
    ("life_expectancy",    "Life Expectancy (years)"),
    ("infant_mortality",   "Infant Mortality (per 1000)"),
    ("maternal_mortality", "Maternal Mortality (per 100k)"),
    ("gdp_per_capita",     "GDP per Capita (USD)"),
    ("health_expenditure", "Health Expenditure (% GDP)"),
    ("vaccination_dtp3",   "DTP3 Vaccination (%)"),
]

colors = plt.cm.Set2.colors

for ax, (col, label), color in zip(axes, plot_cols, colors):
    data = df[col].dropna()
    ax.hist(data, bins=18, color=color, alpha=0.65, edgecolor="white", density=True)
    kde_x = np.linspace(data.min(), data.max(), 200)
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(data)
    ax.plot(kde_x, kde(kde_x), color="black", linewidth=1.5)
    ax.axvline(data.mean(),   color="red",   linestyle="--", linewidth=1, label="mean")
    ax.axvline(data.median(), color="green", linestyle=":",  linewidth=1, label="median")
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle("Distribution of Africa Health Indicators", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("distributions.png")
plt.show()

In [ ]:
# Boxplot vs Violin — life expectancy by region
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

region_order = df.groupby("region")["life_expectancy"].median().sort_values().index.tolist()

# Box plot
sns.boxplot(data=df, x="region", y="life_expectancy",
            order=region_order, palette="Set2", width=0.5, ax=axes[0])
axes[0].set_title("Box Plot", fontsize=12)
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)

# Violin plot (shows distribution shape)
sns.violinplot(data=df, x="region", y="life_expectancy",
               order=region_order, palette="Set2",
               inner="quart", ax=axes[1])
axes[1].set_title("Violin Plot", fontsize=12)
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=25)

for ax in axes:
    ax.set_ylabel("Life Expectancy (years)")

plt.suptitle("Life Expectancy by Region — Box vs Violin", fontsize=13)
plt.tight_layout()
plt.savefig("box_violin.png")
plt.show()

In [ ]:
# Strip plot with mean markers — shows every data point
fig, ax = plt.subplots(figsize=(11, 5))

sns.stripplot(data=df, x="region", y="life_expectancy",
              order=region_order, palette=list(REGION_COLORS.values()),
              jitter=0.15, alpha=0.75, size=7, ax=ax)

# Overlay region means as horizontal markers
for i, region in enumerate(region_order):
    mean_val = df[df["region"] == region]["life_expectancy"].mean()
    ax.hlines(mean_val, i - 0.3, i + 0.3, colors="black", linewidth=2.5)

ax.set_title("Life Expectancy by Region (each point = one country)", fontsize=12)
ax.set_xlabel("")
ax.set_ylabel("Life Expectancy (years)")
ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig("strip_plot.png")
plt.show()

---
## 3. Correlation Heatmaps

In [ ]:
feat_cols = [
    "life_expectancy", "infant_mortality", "maternal_mortality",
    "hiv_prevalence", "health_expenditure", "gdp_per_capita",
    "vaccination_dtp3", "water_access", "malaria_incidence",
    "physicians_per1k"
]
feat_cols = [c for c in feat_cols if c in df.columns]
corr = df[feat_cols].corr()

# Mask upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", linewidths=0.5,
    cmap="RdYlGn", vmin=-1, vmax=1, center=0,
    square=True, ax=ax,
    annot_kws={"size": 8},
    cbar_kws={"shrink": 0.8, "label": "Pearson r"}
)
ax.set_title("Correlation Matrix — Africa Health Indicators",
             fontsize=13, fontweight="bold", pad=15)
ax.tick_params(axis="x", rotation=35, labelsize=9)
ax.tick_params(axis="y", rotation=0,  labelsize=9)

plt.tight_layout()
plt.savefig("correlation_heatmap.png")
plt.show()

In [ ]:
# Heatmap of regional averages — normalised
metrics = ["life_expectancy","infant_mortality","maternal_mortality",
           "vaccination_dtp3","water_access","health_expenditure"]
metrics = [m for m in metrics if m in df.columns]

region_means = df.groupby("region")[metrics].mean()

# Normalise each column to 0-1 for comparability
region_norm = (region_means - region_means.min()) / (region_means.max() - region_means.min())

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(region_norm, annot=region_means.round(1), fmt=".1f",
            cmap="YlOrRd", linewidths=0.5,
            cbar_kws={"label": "Normalised (0=min, 1=max)"},
            ax=ax, annot_kws={"size": 9})
ax.set_title("Regional Health Profile (colour = relative position, number = actual value)",
             fontsize=11)
ax.tick_params(axis="x", rotation=30, labelsize=9)
ax.tick_params(axis="y", rotation=0,  labelsize=9)
plt.tight_layout()
plt.savefig("regional_heatmap.png")
plt.show()

---
## 4. Before / After Comparisons

Visualising the effect of data transformations is critical for communicating preprocessing decisions.

In [ ]:
col  = "maternal_mortality"
raw  = df[col].dropna().values
wins = np.array(winsorize(raw, limits=[0.05, 0.05]))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Row 1: Histograms
for ax, (data, label, color) in zip(
    axes[0],
    [(raw,  "Before: Raw",       "#E74C3C"),
     (wins, "After: Winsorized (5%)", "#27AE60")]
):
    ax.hist(data, bins=22, color=color, alpha=0.72, edgecolor="white", density=True)
    ax.axvline(data.mean(),   color="navy",  linestyle="--", label=f"mean={data.mean():.0f}")
    ax.axvline(np.median(data), color="gold", linestyle=":"  , label=f"median={np.median(data):.0f}")
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Maternal Mortality (per 100k)")
    ax.legend(fontsize=8)

# Row 2: Box plots
for ax, (data, label) in zip(
    axes[1],
    [(raw,  "Before"), (wins, "After")]
):
    bp = ax.boxplot(data, vert=True, patch_artist=True, widths=0.5,
                    boxprops=dict(facecolor="#AED6F1"),
                    medianprops=dict(color="navy", linewidth=2),
                    flierprops=dict(marker="o", color="crimson", markersize=5))
    ax.set_title(f"Box Plot — {label}", fontsize=11)
    ax.set_xticks([])
    stats_text = f"Mean={data.mean():.0f}\nStd={data.std():.0f}\nMax={data.max():.0f}"
    ax.text(1.35, data.mean(), stats_text, fontsize=8, va="center",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.suptitle("Maternal Mortality: Before & After Winsorization",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("before_after_winsorize.png")
plt.show()

In [ ]:
# Log transformation effect
raw_gdp = df["gdp_per_capita"].values
log_gdp = np.log1p(raw_gdp)

from scipy import stats as st

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Histograms
for ax, data, title, color in [
    (axes[0,0], raw_gdp, "GDP per Capita (Raw)",      "#E74C3C"),
    (axes[0,1], log_gdp, "Log(GDP per Capita + 1)",   "#2ECC71"),
]:
    ax.hist(data, bins=20, color=color, alpha=0.7, edgecolor="white", density=True)
    ax.set_title(title, fontsize=11)

# Q-Q plots
for ax, data, title in [
    (axes[1,0], raw_gdp, "Q-Q: Raw GDP"),
    (axes[1,1], log_gdp, "Q-Q: Log GDP"),
]:
    st.probplot(data, plot=ax)
    ax.set_title(title, fontsize=11)
    ax.get_lines()[0].set(markersize=4, alpha=0.6)

plt.suptitle("Log Transformation: Normalising Skewed GDP Data", fontsize=13)
plt.tight_layout()
plt.savefig("log_transform.png")
plt.show()

---
## 5. Seaborn Statistical Plots

In [ ]:
# Pair plot — relationships between all variable pairs, coloured by region
pair_cols = ["life_expectancy", "infant_mortality",
             "gdp_per_capita",  "vaccination_dtp3"]

g = sns.pairplot(
    df[pair_cols + ["region"]].dropna(),
    hue="region", palette=list(REGION_COLORS.values()),
    corner=True, plot_kws={"alpha": 0.6, "s": 40},
    diag_kind="kde"
)
g.fig.suptitle("Pairplot — Africa Health Key Indicators by Region",
               y=1.02, fontsize=13)
plt.savefig("pairplot.png")
plt.show()

In [ ]:
# FacetGrid: regression line per region
g = sns.FacetGrid(df.dropna(), col="region", col_wrap=3,
                  height=3.5, sharey=False, sharex=False)
g.map_dataframe(sns.regplot, x="gdp_per_capita", y="life_expectancy",
                scatter_kws={"s": 30, "alpha": 0.7}, line_kws={"color": "crimson"})
g.set_axis_labels("GDP per Capita (USD)", "Life Expectancy (years)")
g.set_titles(col_template="{col_name}")
g.fig.suptitle("GDP vs. Life Expectancy by Region", y=1.03, fontsize=13)
plt.savefig("facet_regression.png")
plt.show()

In [ ]:
# Grouped bar chart — multiple metrics by region
metrics_norm = ["life_expectancy", "vaccination_dtp3", "water_access"]
df_melt = df.melt(id_vars=["region"], value_vars=metrics_norm,
                  var_name="indicator", value_name="value")

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=df_melt, x="region", y="value", hue="indicator",
            palette="Set2", errorbar="sd", capsize=0.05, ax=ax)
ax.set_title("Health Indicators by Region (mean ± SD)", fontsize=12)
ax.set_xlabel("")
ax.set_ylabel("Value")
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Indicator", fontsize=9)
plt.tight_layout()
plt.savefig("grouped_bar.png")
plt.show()

---
## 6. Multi-Panel Dashboard

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel A: Scatter GDP vs LE ──────────────────────────────────────
ax_a = fig.add_subplot(gs[0, :2])
for region, color in REGION_COLORS.items():
    sub = df[df["region"] == region]
    ax_a.scatter(sub["gdp_per_capita"], sub["life_expectancy"],
                 c=color, label=region, s=55, alpha=0.8, edgecolors="white")
ax_a.set_title("A. GDP per Capita vs. Life Expectancy", fontweight="bold")
ax_a.set_xlabel("GDP per Capita (USD)")
ax_a.set_ylabel("Life Expectancy (years)")
ax_a.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
ax_a.legend(fontsize=7, ncol=2)

# ── Panel B: Vaccination donut ─────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 2])
reg_vax = df.groupby("region")["vaccination_dtp3"].mean().sort_values()
wedges, texts, autotexts = ax_b.pie(
    reg_vax, labels=None, autopct="%1.0f%%",
    colors=list(REGION_COLORS.values()),
    wedgeprops=dict(width=0.55), startangle=90,
    textprops={"fontsize": 8}
)
ax_b.set_title("B. Vaccination Coverage\nby Region", fontweight="bold")
ax_b.legend(reg_vax.index, loc="lower center", fontsize=6,
            bbox_to_anchor=(0.5, -0.3), ncol=2)

# ── Panel C: Infant mortality histogram ────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])
ax_c.hist(df["infant_mortality"].dropna(), bins=18, color="#E74C3C",
          alpha=0.75, edgecolor="white")
ax_c.set_title("C. Infant Mortality Distribution", fontweight="bold")
ax_c.set_xlabel("per 1000 live births")

# ── Panel D: Boxplot HIV by region ─────────────────────────────────
ax_d = fig.add_subplot(gs[1, 1:])
region_order = df.groupby("region")["hiv_prevalence"].median().sort_values().index.tolist()
sns.boxplot(data=df, x="region", y="hiv_prevalence",
            order=region_order, palette=list(REGION_COLORS.values()),
            width=0.5, ax=ax_d)
ax_d.set_title("D. HIV Prevalence by Region", fontweight="bold")
ax_d.set_xlabel("")
ax_d.tick_params(axis="x", rotation=20, labelsize=8)

# ── Panel E: Correlation heatmap (mini) ────────────────────────────
ax_e = fig.add_subplot(gs[2, :])
mini_cols = ["life_expectancy","infant_mortality","gdp_per_capita",
             "health_expenditure","vaccination_dtp3","water_access"]
mini_cols = [c for c in mini_cols if c in df.columns]
corr_mini = df[mini_cols].corr()
mask_mini = np.triu(np.ones_like(corr_mini, dtype=bool))
sns.heatmap(corr_mini, mask=mask_mini, annot=True, fmt=".2f",
            cmap="RdYlGn", vmin=-1, vmax=1, center=0,
            linewidths=0.4, ax=ax_e, annot_kws={"size": 8},
            cbar_kws={"shrink": 0.6})
ax_e.set_title("E. Correlation Matrix", fontweight="bold")
ax_e.tick_params(axis="x", rotation=20, labelsize=9)

fig.suptitle("Africa Health Indicators Dashboard — 2022",
             fontsize=16, fontweight="bold", y=1.01)
plt.savefig("dashboard.png", bbox_inches="tight")
plt.show()

---
## 7. Exercises

### Exercise 1: Distribution Analysis
Create a 1×2 figure comparing the distribution of `malaria_incidence` before and after a log transformation.  
Include histogram, KDE, and Q-Q plot for each.

In [ ]:
# Exercise 1


### Exercise 2: Custom Heatmap
Create a heatmap showing the **percentage of countries** in each region that fall into each life expectancy band (Low / Moderate / High / Very High).  
Annotate each cell with both the percentage and the count.

In [ ]:
# Exercise 2


### Exercise 3: Before/After Imputation
For `physicians_per1k`, create a 2-row figure:  
Row 1: Histogram (before) | Histogram (after median imputation)  
Row 2: Box plots for the same two states  
Add annotations showing mean and std for each panel.

In [ ]:
# Exercise 3


---
## Summary

| Chart | Best for | Library |
|---|---|---|
| Histogram | Distribution shape | plt / sns.histplot |
| KDE | Smooth density estimate | sns.kdeplot |
| Box plot | Quartiles + outliers | sns.boxplot |
| Violin | Distribution + quartiles | sns.violinplot |
| Strip plot | All data points | sns.stripplot |
| Heatmap | Matrix values | sns.heatmap |
| Pair plot | All pairwise relationships | sns.pairplot |
| FacetGrid | Same chart per group | sns.FacetGrid |
| GridSpec | Complex multi-panel layout | matplotlib.gridspec |

**Next:** Notebook 4 — End-to-End 8-Step Pipeline